In [28]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [29]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] > 5) &
    (df1["works_count"] < 5000)
].tolist()

subset = df1.loc[indexes, "author_id"]
print(subset)

0      https://openalex.org/A5022704573
1      https://openalex.org/A5079294298
2      https://openalex.org/A5036764962
3      https://openalex.org/A5038833789
4      https://openalex.org/A5091326079
                     ...               
928    https://openalex.org/A5100951653
929    https://openalex.org/A5064802580
930    https://openalex.org/A5060299718
931    https://openalex.org/A5018297686
932    https://openalex.org/A5093454296
Name: author_id, Length: 766, dtype: object


In [70]:
# Sociology, Psychology, Economics, Political Science
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
# Mathematics, Physics, Computer Science
quant_concepts = ["C33923547", "C121332964", "C41008148"]

# Build the concept filter string
# Works must have ANY CSS concept AND ANY quant concept
concept_filter = f"concept.id:{'|'.join(css_concepts)}+concept.id:{'|'.join(quant_concepts)}"

rows2 = []
rows3 = []


for author_id in subset:
    
    # Handle pagination
    page = 1
    has_more = True
    unique_author_ids = set()
    
    while has_more:
        # Fixed filter syntax (removed .search)
        params = {
            "filter": f"corresponding_author_ids:{author_id},cited_by_count:>10,authors_count:<10,{concept_filter}",
            "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
            "per-page": 200,
            "page": page,
            "api_key": API_KEY
        }
        
        
        response = requests.get('https://api.openalex.org/works', params=params)
        data = response.json()
        print(data)
            
            
            
        if not data.get("results"):
            break  # No more results
            
        results = data["results"]
        print(f"  Page {page}: Found {len(results)} works")
        
        for item in results:
            rows2.append({
                "id": item.get('id'),
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "corresponding_author_ids": item.get("corresponding_author_ids"),
            })
            rows3.append({
                "id": item.get('id'),
                "title": item.get('title'),
                "abstract_index": item.get('abstract_inverted_index')  # Fixed key name
            })
        
        # Check if this was the last page
        if len(results) < 200:
            has_more = False
        else:
            page += 1
            sleep(0.1)
        
            

# Be nice to the API - delay between authors
    sleep(0.5)

# Create DataFrames
df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print(f"\nComplete!")
print(f"df2: {len(df2)} rows")
print(f"df3: {len(df3)} rows")


{'error': 'Rate limit exceeded', 'message': 'Insufficient budget. This request costs $0.0001 but you only have $0 remaining. Resets at midnight UTC. Need more? Add funds at https://openalex.org/pricing', 'retryAfter': 18813, 'costUsd': 0.0001, 'dailyRemainingUsd': 0, 'prepaidRemainingUsd': 0, 'creditsRequired': 1, 'creditsRemaining': 0, 'onetimeCreditsRemaining': 0}
{'error': 'Rate limit exceeded', 'message': 'Insufficient budget. This request costs $0.0001 but you only have $0 remaining. Resets at midnight UTC. Need more? Add funds at https://openalex.org/pricing', 'retryAfter': 18812, 'costUsd': 0.0001, 'dailyRemainingUsd': 0, 'prepaidRemainingUsd': 0, 'creditsRequired': 1, 'creditsRemaining': 0, 'onetimeCreditsRemaining': 0}
{'error': 'Rate limit exceeded', 'message': 'Insufficient budget. This request costs $0.0001 but you only have $0 remaining. Resets at midnight UTC. Need more? Add funds at https://openalex.org/pricing', 'retryAfter': 18812, 'costUsd': 0.0001, 'dailyRemainingUsd

KeyboardInterrupt: 

In [68]:
# If corresponding_author_ids contains lists

papers_with_multiple_corresponding = df2[df2["corresponding_author_ids"].str.len() > 1]
papers_with_multiple_corresponding

KeyError: 'corresponding_author_ids'

In [36]:
df2.to_csv("works_ids.csv", index=False)
df3.to_csv("works_titles_abstracts.csv", index=False)

## Co-Authors)

In [ ]:
# co_author_ids = []
# corresponding_unique_author_ids = df2["corresponding_author_ids"].explode().unique()
# for author_id in corresponding_unique_author_ids:
#     if author_id not in df1["author_id"].values:
#         co_author_ids.append(author_id)

# co_author_ids

['https://openalex.org/A5042853862',
 'https://openalex.org/A5060975310',
 'https://openalex.org/A5026111773',
 'https://openalex.org/A5045223218',
 'https://openalex.org/A5084138506',
 'https://openalex.org/A5085123886',
 'https://openalex.org/A5043743069',
 'https://openalex.org/A5058463849',
 'https://openalex.org/A5103100312',
 'https://openalex.org/A5030318475',
 'https://openalex.org/A5041645225',
 'https://openalex.org/A5048665132',
 'https://openalex.org/A5047081667',
 'https://openalex.org/A5073615402',
 'https://openalex.org/A5100722039',
 'https://openalex.org/A5001911082',
 'https://openalex.org/A5039573704',
 'https://openalex.org/A5076336127',
 'https://openalex.org/A5075225440',
 'https://openalex.org/A5112530748',
 'https://openalex.org/A5003014116',
 'https://openalex.org/A5002765396',
 'https://openalex.org/A5040476342',
 'https://openalex.org/A5052664502',
 'https://openalex.org/A5100655276',
 'https://openalex.org/A5006577991',
 'https://openalex.org/A5089519579',
 

In [ ]:
# rows4 = []
# for coauthor_id in co_author_ids:
#     URL = "https://api.openalex.org/authors"

#     params = {
#         "filter": f"id:{coauthor_id}",
#         "select": "display_name,works_count,works_api_url,last_known_institutions,summary_stats",
#         "api_key": API_KEY
#     }

#     response = requests.get(URL, params=params)
#     data = response.json()
    
#     if data.get("results"):
#         data = data["results"][0]
#         h_index = data.get("summary_stats", {}).get("h_index")

#         # Get first institution country code
#         institutions = data.get("last_known_institutions", [])
#         country_code = institutions[0].get("country_code") if institutions else None
#         rows4.append({
#                 "display_name": data.get("display_name"),
#                 "works_count": data.get("works_count"),
#                 "works_api_url": data.get("works_api_url"),
#                 "h_index": h_index,
#                 "country_code": country_code
#             })
    
    
# df4 = pd.DataFrame(rows4)
# df4.to_csv("co_authors_data.csv", index=False)

## Co-author Works

In [ ]:
# rows5 = []
# rows6 = []
# for co_author_id in co_author_ids:
#     # Handle pagination
#     page = 1
#     has_more = True
#     unique_author_ids = set()
    
#     while has_more:
#         # Fixed filter syntax (removed .search)
#         params = {
#             "filter": f"corresponding_author_ids:{co_author_id},cited_by_count:>10,authors_count:<10",
#             "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
#             "per-page": 200,
#             "page": page,
#             "api_key": API_KEY
#         }
        
        
#         response = requests.get('https://api.openalex.org/works', params=params)
#         data = response.json()
            
            
            
#         if not data.get("results"):
#             break  # No more results
            
#         results = data["results"]
#         for work_id in results[0].get("id"):
#             if work_id in df2["id"].values:
#                 break
#         else:
#             pass
#         only_co_and_author = []
#         corresponding_author_ids = item.get("corresponding_author_ids")
#         for corresponding_author_id in corresponding_author_ids:
#             if corresponding_author_id in co_author_ids:
#                 only_co_and_author.append(corresponding_author_id)
#             elif corresponding_author_id in subset:
#                 only_co_and_author.append(corresponding_author_id)

            

        
#         for item in results:
#             rows5.append({
#                 "id": item.get('id'),
#                 "publication_year": item.get("publication_year"),
#                 "cited_by_count": item.get("cited_by_count"),
#                 "corresponding_author_ids": only_co_and_author,
#             })
#             rows6.append({
#                 "id": item.get('id'),
#                 "title": item.get('title'),
#                 "abstract_index": item.get('abstract_inverted_index')  
#             })
        
#         # Check if this was the last page
#         if len(results) < 200:
#             has_more = False
#         else:
#             sleep(0.1)

# # Be nice to the API - delay between authors
#     sleep(0.5)

# # Create DataFrames
# df5 = pd.DataFrame(rows5)
# df6 = pd.DataFrame(rows6)

# print(f"\nComplete!")
# print(f"df5: {len(df5)} rows")
# print(f"df6: {len(df6)} rows")



Complete!
df5: 1168380 rows
df6: 1168380 rows


In [66]:
df5.to_csv("co_authors_works_ids.csv", index=False)
df6.to_csv("co_authors_works_titles_abstracts.csv", index=False)